# 04a · Visual classification of galaxies: input lists

Prepares the object lists for the visual inspection of SDSS images (with the SDSS image-list tool) that decides whether each photometric source is a galaxy, and converts the inspection result into a machine-readable table. The result is merged into the master catalog by `07_Photometry_Flag_update.ipynb`.

**Input**
- `A2199_mastercat_intermediate_file0.csv` – merged catalog from `03_merge_mastercat.py`
- `../AllHeCS_VAC_updated.csv` – HeCS-omnibus cluster catalog

**Files**
- `04b_galaxy_vis_classify_SDSSimglist_input.txt` – objid, RA, Dec of the candidates, sorted by r magnitude (image-list input)
- `04c_galaxy_vis_classify_SDSSimglist_template.txt` – result template (all flags 1); copied to `04c_galaxy_vis_classify_SDSSimglist_result.txt` and filled in by hand (galaxyflag: 0 = not a galaxy, 1 = galaxy, 2 = ambiguous)
- `04d_A2199galaxy_visual_classification_result.csv` – the inspection result as CSV
- `04e_galaxy_vis_classify_wrong_SDSSimglist_input.txt` – objects flagged as non-galaxy or ambiguous, for a second look

Only the notebook is tracked in the repository; the lists and results are not.

In [1]:
# ============================================================================
# Setup
# ============================================================================
import numpy as np
import pandas as pd

# Show every column when a DataFrame is displayed
pd.set_option('display.max_columns', None)

In [3]:
hecs_vac_data = pd.read_csv('../../DATA/AllHeCS_VAC_updated.csv')
Z_CLID = hecs_vac_data[hecs_vac_data['CLID'] == 'A2199']['Z'].values[0]

# Merged catalog (photometry + redshifts from all sources)
df0 = pd.read_csv('./A2199_mastercat_intermediate_file0.csv')

In [4]:
# Galactic-extinction-corrected model and Petrosian magnitudes (suffix "_0")
for band in ['u', 'g', 'r', 'i', 'z']:
    df0[f'p_modelmag_{band}_0'] = df0[f'p_modelmag_{band}'] - df0[f'p_extinction_{band}']
for band in ['u', 'g', 'r', 'i', 'z']:
    df0[f'p_petromag_{band}_0'] = df0[f'p_petromag_{band}'] - df0[f'p_extinction_{band}']

In [5]:
df0['grmod'] = df0['p_modelmag_g_0'] - df0['p_modelmag_r_0']

## Candidate galaxies for visual inspection

In [6]:
# Candidates: extended by the SDSS star/galaxy separator (probPSF != 1: 0 = extended, 1 = point source),
# with valid photometry and a plausible colour
galaxy_mask = (df0['p_probpsf'] != 1) & (df0['phot_source'] != 'wrong') & (df0['grmod'] >= -0.25) & (df0['grmod'] <= 2.5)

df = df0[galaxy_mask]

In [7]:
# How many objects rejected by the Strauss et al. (2002) star/galaxy criteria
# nevertheless have a measured redshift (within 35 arcmin, r <= 21)?
print(len(df0[(~galaxy_mask) & (df0['z_tot_z'] > 0) & (df0['p_radgal'] < 35) & (df0['p_petromag_r_0'] <= 21.0)]))
print(len(df0[(~galaxy_mask) & (df0['p_radgal'] < 35) & (df0['p_petromag_r_0'] <= 21.0)]))
print(len(df0[(df0['z_tot_z'] > 0) & (df0['p_radgal'] < 35) & (df0['p_petromag_r_0'] <= 21.0)]))
print(135/2474 * 100)      # percentage typed in from an earlier run of the counts above

115
5523
2452
5.456750202101859


Image-list input for the visual classification of every candidate galaxy (objid RA Dec, brightest first).

In [ ]:
with open('04b_galaxy_vis_classify_SDSSimglist_input.txt', 'w') as f:
    temp = df.sort_values(by='p_modelmag_r', ascending=True)
    for idx, row in temp.iterrows():
        f.write(f"{row['p_objid']} {row['p_ra']:.6f} {row['p_dec']:.6f}\n")

Template for the classification result. Every object starts as a galaxy (flag 1). Copy the template to `04c_galaxy_vis_classify_SDSSimglist_result.txt` and edit the flags by hand during the inspection; the template itself is never read back, so re-running this notebook cannot overwrite the inspection result.

In [ ]:
with open('04c_galaxy_vis_classify_SDSSimglist_template.txt', 'w') as f:
    f.write("p_objid,p_ra,p_dec,galaxyflag\n")
    f.write("galaxyflag 0 = Not Galaxy 1 = Galaxy 2 = Ambiguous\n")
    temp = df.sort_values(by='p_modelmag_r', ascending=True)
    for idx, row in temp.iterrows():
        f.write(f"{row['p_objid']},{row['p_ra']:.6f},{row['p_dec']:.6f}, 1\n")

Convert the filled-in result to CSV (the second line of the text file is a note and is skipped).

In [ ]:
file_path = "./04c_galaxy_vis_classify_SDSSimglist_result.txt"
vis = pd.read_csv(file_path, sep=',', skiprows=[1])
vis['galaxyflag'] = vis['galaxyflag'].astype(int)
vis.to_csv("./04d_A2199galaxy_visual_classification_result.csv", sep=',', index=False)

### Notes on the visual classification

- Objects that are much fainter than their catalogued magnitude suggests, fragments of a larger galaxy, or sources whose photometry is contaminated by a nearby bright object, as well as objects that are clearly not galaxies, are flagged `galaxyflag = 0`.
- An object that looks like one of the cases above but has a measured redshift is kept as a galaxy.
- Two objects with a redshift are nevertheless rejected because they are unambiguous fragments of a larger galaxy (confirmed with H. S. Hwang): 1237659326566039624 and 1237659326566039622.

In [103]:
# Objects flagged as non-galaxy or ambiguous
vis_wrong = vis[vis['galaxyflag'] != 1].copy()

In [104]:
vis_wrong

,p_objid,p_ra,p_dec,galaxyflag
0,1237659330852094049,247.137002,39.776718,0
1,1237659326029168686,246.810020,39.888695,0
2,1237659330315485207,247.160630,39.082430,0
3,1237659325492297792,246.414437,39.580813,0
4,1237659330315354181,246.938737,39.296788,0
...,...,...,...,...
5913,1237659330315289909,246.905544,39.456780,2
6048,1237659325492560383,246.792528,39.047635,2
6121,1237659330315223094,246.846982,39.572034,2
6220,1237659330852225089,247.414732,39.597132,2


In [105]:
# Image-list input for a second look at the flagged objects
with open('04e_galaxy_vis_classify_wrong_SDSSimglist_input.txt', 'w') as f:
    temp = vis_wrong
    temp['p_objid'] = temp['p_objid'].astype(str)
    for idx, row in temp.iterrows():
        f.write(f"{row['p_objid']} {row['p_ra']:.6f} {row['p_dec']:.6f}\n")